# ⚙️ Stage 12: MLOps & Model Deployment
## Project: End-to-End Salary Predictor Pipeline

**Goal:** Build a salary prediction model, track experiments with MLflow,  
orchestrate retraining with Prefect, and deploy as a FastAPI + Docker service.

| Component | Technology |
|---|---|
| Data | Synthetic Stack Overflow Developer Survey (90K rows) |
| Models | Linear Regression · Random Forest · XGBoost |
| Experiment Tracking | MLflow (runs, metrics, artifacts, model registry) |
| Orchestration | Prefect (retraining pipeline as a flow) |
| API Layer | FastAPI (REST endpoint with Pydantic validation) |
| Containerisation | Docker + docker-compose |
| CI/CD | GitHub Actions workflow |
| Monitoring | Data drift detection + model performance tracking |

---
🗣 **Tamil:** Model build பண்றது போதாது — production-ல் வேலை செய்ய வைப்பது MLOps.  
MLflow experiments track செய்யும். Prefect retraining automate செய்யும். FastAPI + Docker deploy செய்யும்.

## 📦 1. Imports & Configuration

In [1]:
import os, json, time, warnings, shutil, subprocess, textwrap
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
warnings.filterwarnings('ignore')

# ML
from sklearn.model_selection   import train_test_split, cross_val_score, KFold
from sklearn.preprocessing     import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose           import ColumnTransformer
from sklearn.pipeline          import Pipeline
from sklearn.linear_model      import Ridge
from sklearn.ensemble          import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics           import mean_squared_error, mean_absolute_error, r2_score
from sklearn.impute            import SimpleImputer
from sklearn.inspection        import permutation_importance
import xgboost as xgb

# MLflow
import mlflow
import mlflow.sklearn
from mlflow.models             import infer_signature

# Prefect
from prefect                   import flow, task
from prefect.logging           import get_run_logger

# Plotting
plt.rcParams.update({
    'figure.facecolor':'#080812','axes.facecolor':'#10101e',
    'axes.edgecolor':'#35355a','text.color':'#dcdcff',
    'axes.labelcolor':'#dcdcff','xtick.color':'#8888b0',
    'ytick.color':'#8888b0','grid.color':'#202038','grid.alpha':0.5,
    'legend.facecolor':'#181830','legend.edgecolor':'#404060',
})
C1,C2,C3,C4 = '#00d4ff','#00ff9f','#ffd700','#ff6b6b'

MLFLOW_URI   = './mlruns'
EXPERIMENT   = 'salary-predictor'
MODEL_NAME   = 'SalaryPredictor'
mlflow.set_tracking_uri(MLFLOW_URI)
mlflow.set_experiment(EXPERIMENT)

print("✅ All imports successful")
print(f"   MLflow     : {mlflow.__version__}")
print(f"   XGBoost    : {xgb.__version__}")
print(f"   Prefect    : {__import__('prefect').__version__}")
print(f"   FastAPI    : {__import__('fastapi').__version__}")
print(f"   Tracking @ : {MLFLOW_URI}")

MlflowException: The filesystem tracking backend (e.g., './mlruns') is in maintenance mode and will not receive further updates. Please migrate to a database backend (e.g., 'sqlite:///mlflow.db') to access the latest MLflow features. The `mlflow migrate-filestore` tool migrates your existing data losslessly. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance. If the filesystem backend is required for your workflow, set `MLFLOW_ALLOW_FILE_STORE=true` to opt out of this exception.

## 📊 2. Synthetic Stack Overflow Developer Survey Dataset

Recreating the SO Developer Survey schema with **90,000 respondents** and realistic salary distributions.

| Feature | Type | Description |
|---|---|---|
| `YearsCodePro` | numeric | Years of professional coding experience |
| `Country` | categorical | Country of residence |
| `EdLevel` | categorical | Highest education level |
| `DevType` | categorical | Developer type (fullstack, backend, etc.) |
| `OrgSize` | categorical | Organisation size |
| `Employment` | categorical | Employment type |
| `Age` | numeric | Age of respondent |
| `WorkWeekHrs` | numeric | Weekly working hours |
| `RemoteWork` | categorical | Remote work arrangement |
| `LanguagesUsed` | numeric | Number of languages used |
| `ConvertedCompYearly` | numeric | **Target — Annual USD salary** |

🗣 **Tamil:** Stack Overflow developer survey-ஐ synthetic-ஆக உருவாக்குகிறோம். 90,000 developers-இன் salary, experience, country, education data.

In [ ]:
np.random.seed(42)
N = 90_000

# ── Categorical features ───────────────────────────────────────────────────
COUNTRIES   = ['USA','India','Germany','UK','Canada','Australia',
                'Brazil','France','Netherlands','Poland']
COUNTRY_P   = [0.30,0.18,0.08,0.07,0.06,0.05,0.05,0.04,0.04,0.03]
EXTRA_SHARE = 1 - sum(COUNTRY_P)
COUNTRY_P[-1] += EXTRA_SHARE   # absorb rounding

EDLEVEL = ['Bachelor','Master','Some college','PhD','Bootcamp','No degree']
EDLEVEL_P = [0.42,0.28,0.12,0.09,0.06,0.03]

DEVTYPE = ['Full-stack','Backend','Frontend','Data scientist',
           'DevOps','Mobile','Embedded','ML engineer']
DEVTYPE_P = [0.28,0.22,0.14,0.12,0.09,0.07,0.04,0.04]

ORGSIZE = ['1-9','10-99','100-999','1000-4999','5000+']
ORGSIZE_P = [0.15,0.25,0.28,0.17,0.15]

EMPLOYMENT = ['Full-time','Independent contractor','Part-time','Freelancer']
EMPLOYMENT_P = [0.75,0.14,0.07,0.04]

REMOTE = ['In-person','Hybrid','Fully remote']
REMOTE_P = [0.35,0.42,0.23]

def pick(choices, probs, n):
    probs = np.array(probs, dtype=float)
    probs = probs / probs.sum()
    return np.random.choice(choices, n, p=probs)

country    = pick(COUNTRIES,   COUNTRY_P,    N)
edlevel    = pick(EDLEVEL,     EDLEVEL_P,    N)
devtype    = pick(DEVTYPE,     DEVTYPE_P,    N)
orgsize    = pick(ORGSIZE,     ORGSIZE_P,    N)
employment = pick(EMPLOYMENT,  EMPLOYMENT_P, N)
remote     = pick(REMOTE,      REMOTE_P,     N)

# ── Numeric features ───────────────────────────────────────────────────────
years_exp    = np.random.gamma(shape=2.2, scale=4.0, size=N).clip(0, 40)
age          = (22 + years_exp + np.random.normal(0, 3, N)).clip(18, 70)
work_hrs     = np.random.normal(42, 8, N).clip(20, 80)
langs_used   = np.random.poisson(3.5, N).clip(1, 15)

# ── Salary model (log-normal with feature effects) ─────────────────────────
BASE = 65_000

country_mult = {
    'USA':1.85,'Germany':1.30,'UK':1.25,'Canada':1.20,
    'Australia':1.18,'Netherlands':1.22,'France':1.15,
    'Poland':0.70,'Brazil':0.45,'India':0.38,
}
ed_mult = {'PhD':1.30,'Master':1.18,'Bachelor':1.05,
           'Some college':0.92,'Bootcamp':0.88,'No degree':0.80}
dev_mult = {'ML engineer':1.35,'Data scientist':1.28,'Backend':1.12,
            'DevOps':1.15,'Full-stack':1.08,'Frontend':0.95,
            'Mobile':1.00,'Embedded':1.05}
org_mult = {'5000+':1.22,'1000-4999':1.12,'100-999':1.00,
            '10-99':0.88,'1-9':0.75}
emp_mult = {'Full-time':1.05,'Independent contractor':1.15,
            'Part-time':0.60,'Freelancer':0.90}
remote_mult = {'Fully remote':1.08,'Hybrid':1.03,'In-person':1.00}

c_mult = np.array([country_mult[c] for c in country])
e_mult = np.array([ed_mult[e]      for e in edlevel])
d_mult = np.array([dev_mult[d]     for d in devtype])
o_mult = np.array([org_mult[o]     for o in orgsize])
m_mult = np.array([emp_mult[m]     for m in employment])
r_mult = np.array([remote_mult[r]  for r in remote])

exp_effect  = 1 + 0.040 * years_exp - 0.0006 * years_exp**2
lang_effect = 1 + 0.025 * langs_used

salary_raw  = (BASE * c_mult * e_mult * d_mult * o_mult * m_mult
               * r_mult * exp_effect * lang_effect)
noise       = np.random.lognormal(0, 0.22, N)
salary      = (salary_raw * noise).clip(10_000, 500_000).astype(int)

# ── Inject missing values (~5%) ────────────────────────────────────────────
for col_arr in [years_exp, work_hrs, langs_used]:
    miss_idx = np.random.choice(N, int(N*0.05), replace=False)
    col_arr[miss_idx] = np.nan

df = pd.DataFrame({
    'YearsCodePro'       : years_exp,
    'Country'            : country,
    'EdLevel'            : edlevel,
    'DevType'            : devtype,
    'OrgSize'            : orgsize,
    'Employment'         : employment,
    'RemoteWork'         : remote,
    'Age'                : age.astype(int),
    'WorkWeekHrs'        : work_hrs,
    'LanguagesUsed'      : langs_used,
    'ConvertedCompYearly': salary,
})

df.to_csv('so_survey_synthetic.csv', index=False)
print(f"✅ Dataset: {len(df):,} rows × {df.shape[1]} cols")
print(f"   Salary — median: ${df['ConvertedCompYearly'].median():,.0f}  "
      f"mean: ${df['ConvertedCompYearly'].mean():,.0f}")
print(f"   Missing — YearsCodePro: {df['YearsCodePro'].isna().sum():,}  "
      f"WorkWeekHrs: {df['WorkWeekHrs'].isna().sum():,}")
print()
print(df[['YearsCodePro','Country','DevType','EdLevel',
          'ConvertedCompYearly']].head(6).to_string(index=False))

## 🔍 3. Exploratory Data Analysis

In [ ]:
fig = plt.figure(figsize=(20, 14))
fig.suptitle('📊 Stack Overflow Salary Survey — EDA Dashboard',
             fontsize=16, color=C1, fontweight='bold', y=1.01)
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.48, wspace=0.36)

# Salary distribution
ax1 = fig.add_subplot(gs[0,0:2])
ax1.hist(np.log10(df['ConvertedCompYearly']), bins=60,
         color=C1, alpha=0.85, edgecolor='#080812')
ax1.set_title('Log10(Salary) Distribution', color='#dcdcff')
ax1.set_xlabel('log₁₀(Annual USD)'); ax1.set_ylabel('Count')
ax1.axvline(np.log10(df['ConvertedCompYearly'].median()),
            color=C4, linewidth=2, linestyle='--',
            label=f"Median ${df['ConvertedCompYearly'].median()/1e3:.0f}K")
ax1.legend(framealpha=0.3); ax1.grid(True, alpha=0.3)

# Salary by country
ax2 = fig.add_subplot(gs[0,2])
med_by_country = (df.groupby('Country')['ConvertedCompYearly']
                  .median().sort_values(ascending=True)/1000)
colors_c = [C4 if v > 100 else C1 for v in med_by_country.values]
ax2.barh(med_by_country.index, med_by_country.values,
         color=colors_c, edgecolor='white', linewidth=0.4, alpha=0.85)
ax2.set_title('Median Salary by Country ($K)', color='#dcdcff')
ax2.set_xlabel('Median Salary (USD K)'); ax2.grid(True, alpha=0.3, axis='x')
for v, name in zip(med_by_country.values, med_by_country.index):
    ax2.text(v+1, name, f'${v:.0f}K', va='center', fontsize=8, color='white')

# Experience vs Salary scatter
ax3 = fig.add_subplot(gs[1,0:2])
sample_mask = np.random.choice(len(df), 5000, replace=False)
sc = ax3.scatter(df['YearsCodePro'].iloc[sample_mask],
                 df['ConvertedCompYearly'].iloc[sample_mask]/1000,
                 c=df['Age'].iloc[sample_mask], cmap='plasma',
                 alpha=0.3, s=8)
plt.colorbar(sc, ax=ax3, label='Age')
ax3.set_title('Experience vs Salary (coloured by Age)', color='#dcdcff')
ax3.set_xlabel('Years Experience'); ax3.set_ylabel('Salary ($K)')
ax3.set_ylim(0, 400); ax3.grid(True, alpha=0.3)

# Salary by DevType
ax4 = fig.add_subplot(gs[1,2])
med_dev = (df.groupby('DevType')['ConvertedCompYearly']
           .median().sort_values()/1000)
ax4.barh(med_dev.index, med_dev.values,
         color=C3, edgecolor='white', linewidth=0.4, alpha=0.85)
ax4.set_title('Median Salary by Role ($K)', color='#dcdcff')
ax4.set_xlabel('Median ($K)'); ax4.grid(True, alpha=0.3, axis='x')

# Salary by Education
ax5 = fig.add_subplot(gs[2,0])
med_ed = (df.groupby('EdLevel')['ConvertedCompYearly']
          .median().sort_values(ascending=True)/1000)
ax5.bar(range(len(med_ed)), med_ed.values, color=C2,
        edgecolor='white', linewidth=0.4, alpha=0.85)
ax5.set_xticks(range(len(med_ed)))
ax5.set_xticklabels(med_ed.index, rotation=30, ha='right', fontsize=8)
ax5.set_title('Median Salary by Education ($K)', color='#dcdcff')
ax5.set_ylabel('Median ($K)'); ax5.grid(True, alpha=0.3, axis='y')

# Salary by RemoteWork
ax6 = fig.add_subplot(gs[2,1])
remote_data = [df[df['RemoteWork']==r]['ConvertedCompYearly'].values/1000
               for r in REMOTE]
bp = ax6.boxplot(remote_data, patch_artist=True,
                 medianprops=dict(color=C4,linewidth=2))
for patch, color in zip(bp['boxes'], [C1,C3,C2]):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax6.set_xticklabels(['In-person','Hybrid','Fully Remote'], fontsize=9)
ax6.set_title('Salary by Remote Work', color='#dcdcff')
ax6.set_ylabel('Salary ($K)'); ax6.set_ylim(0, 400)
ax6.grid(True, alpha=0.3, axis='y')

# Correlation heatmap
ax7 = fig.add_subplot(gs[2,2])
num_cols = ['YearsCodePro','Age','WorkWeekHrs','LanguagesUsed','ConvertedCompYearly']
corr = df[num_cols].corr()
sns.heatmap(corr, ax=ax7, cmap='RdBu_r', center=0, annot=True, fmt='.2f',
            annot_kws={'size':9}, cbar_kws={'shrink':0.8},
            xticklabels=['Exp','Age','Hrs','Langs','Salary'],
            yticklabels=['Exp','Age','Hrs','Langs','Salary'])
ax7.set_title('Feature Correlations', color='#dcdcff')

plt.savefig('eda_salary.png', dpi=120, bbox_inches='tight',
            facecolor='#080812', edgecolor='none')
plt.show()
print("✅ EDA dashboard saved → eda_salary.png")

## ⚙️ 4. Scikit-learn Preprocessing Pipeline

Build a reproducible preprocessing pipeline that handles:
- **Median imputation** for missing numerics
- **OneHotEncoding** for categoricals
- **Log transform** on target (salary is right-skewed)

🗣 **Tamil:** Preprocessing pipeline reproducible ஆக இருக்க வேண்டும். Production-ல் train data-இன் same transform test/inference data-க்கும் apply ஆக வேண்டும்.

In [ ]:
TARGET  = 'ConvertedCompYearly'
NUM_COLS = ['YearsCodePro','Age','WorkWeekHrs','LanguagesUsed']
CAT_COLS = ['Country','EdLevel','DevType','OrgSize','Employment','RemoteWork']

X = df[NUM_COLS + CAT_COLS].copy()
y = np.log1p(df[TARGET].values)   # log1p transform — reverse with np.expm1

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42)

print(f"Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}")

# ── Sklearn ColumnTransformer ────────────────────────────────────────────────
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler()),
])
cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ('num', num_transformer, NUM_COLS),
    ('cat', cat_transformer, CAT_COLS),
])

# Fit on train, transform all splits
X_train_pp = preprocessor.fit_transform(X_train)
X_val_pp   = preprocessor.transform(X_val)
X_test_pp  = preprocessor.transform(X_test)

print(f"Preprocessed shape: {X_train_pp.shape}")

# ── Helper metrics (back in original $ space) ────────────────────────────────
def salary_metrics(y_true_log, y_pred_log, label=''):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    rmse   = np.sqrt(mean_squared_error(y_true, y_pred))
    mae    = mean_absolute_error(y_true, y_pred)
    r2     = r2_score(y_true_log, y_pred_log)
    mape   = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    if label:
        print(f"{label:18s}  RMSE=${rmse/1e3:.1f}K  MAE=${mae/1e3:.1f}K  "
              f"R²={r2:.4f}  MAPE={mape:.2f}%")
    return {'rmse':rmse,'mae':mae,'r2':r2,'mape':mape}

print("\nPreprocessing pipeline built ✅")

## 🧪 5. MLflow Experiment Tracking

Train **three models** and track every experiment in MLflow:
- Parameters (hyperparameters)
- Metrics (RMSE, MAE, R², MAPE)
- Artifacts (model files, plots)
- Model registration in the Model Registry

```
mlruns/
  └── salary-predictor/
        ├── run_1  (Ridge Regression)   → params, metrics, model.pkl
        ├── run_2  (Random Forest)      → params, metrics, model.pkl
        └── run_3  (XGBoost)            → params, metrics, model.pkl
```

🗣 **Tamil:** MLflow ஒவ்வொரு experiment-இன் parameters, metrics, artifacts-ஐ track செய்கிறது. யாராவது "ஏன் இந்த model?" என்று கேட்டால் MLflow UI-ல் காட்டலாம்.

In [ ]:
models_config = {
    'Ridge'        : (Ridge(alpha=10.0),
                      {'alpha': 10.0}),
    'RandomForest' : (RandomForestRegressor(
                          n_estimators=200, max_depth=12,
                          min_samples_leaf=5, n_jobs=-1, random_state=42),
                      {'n_estimators':200,'max_depth':12,'min_samples_leaf':5}),
    'XGBoost'      : (xgb.XGBRegressor(
                          n_estimators=400, max_depth=6, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8,
                          min_child_weight=5, reg_alpha=0.1,
                          early_stopping_rounds=30, eval_metric='rmse',
                          random_state=42, n_jobs=-1, verbosity=0),
                      {'n_estimators':400,'max_depth':6,'learning_rate':0.05,
                       'subsample':0.8,'colsample_bytree':0.8}),
}

all_run_ids = {}
all_metrics = {}

for model_name, (model, params) in models_config.items():
    with mlflow.start_run(run_name=model_name) as run:
        mlflow.log_params(params)
        mlflow.set_tag('model_type', model_name)
        mlflow.set_tag('stage', 'training')

        t0 = time.time()
        if model_name == 'XGBoost':
            model.fit(X_train_pp, y_train,
                      eval_set=[(X_val_pp, y_val)], verbose=False)
        else:
            model.fit(X_train_pp, y_train)
        train_time = time.time() - t0

        val_pred  = model.predict(X_val_pp)
        test_pred = model.predict(X_test_pp)

        val_m  = salary_metrics(y_val,  val_pred,  f'{model_name} (Val)')
        test_m = salary_metrics(y_test, test_pred, f'{model_name} (Test)')

        for split, m in [('val',val_m), ('test',test_m)]:
            for k, v in m.items():
                mlflow.log_metric(f'{split}_{k}', v)
        mlflow.log_metric('train_time_sec', train_time)

        # Log model + signature
        sig = infer_signature(X_train_pp[:5], model.predict(X_train_pp[:5]))
        mlflow.sklearn.log_model(model, model_name,
                                 signature=sig,
                                 registered_model_name=MODEL_NAME)

        all_run_ids[model_name] = run.info.run_id
        all_metrics[model_name] = test_m
        all_metrics[model_name]['model']     = model
        all_metrics[model_name]['test_pred'] = test_pred

print("\n✅ All MLflow runs complete")
print(f"   Tracking URI : {mlflow.get_tracking_uri()}")
print(f"   Experiment   : {EXPERIMENT}")
best_model_name = min(all_metrics, key=lambda k: all_metrics[k]['rmse'])
print(f"   Best model   : {best_model_name}  "
      f"(Test RMSE=${all_metrics[best_model_name]['rmse']/1e3:.1f}K)")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('🧪 MLflow Experiment Tracking — Model Comparison',
             fontsize=15, color=C1, fontweight='bold')

model_names = list(all_metrics.keys())
colors_m    = [C1, C2, C3]

# Metrics comparison bar charts
metric_info = [
    ('rmse', 'RMSE ($)',      lambda v: v/1e3,  '$K'),
    ('mae',  'MAE ($)',       lambda v: v/1e3,  '$K'),
    ('r2',   'R² Score',      lambda v: v,      ''),
    ('mape', 'MAPE (%)',      lambda v: v,      '%'),
]
for ax, (key, title, xform, unit) in zip(axes.flat[:4], metric_info):
    vals = [xform(all_metrics[n][key]) for n in model_names]
    bars = ax.bar(model_names, vals, color=colors_m, edgecolor='white',
                  linewidth=0.5, alpha=0.85)
    ax.set_title(title, color='#dcdcff')
    ax.set_ylabel(f'{title} ({unit})' if unit else title)
    ax.grid(True, alpha=0.3, axis='y')
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()*1.01,
                f'{v:.2f}{unit}', ha='center', fontsize=10,
                color='white', fontweight='bold')
    best_idx = np.argmin(vals) if key != 'r2' else np.argmax(vals)
    bars[best_idx].set_edgecolor(C4); bars[best_idx].set_linewidth(2.5)

# Actual vs Predicted — best model
ax5 = axes[1, 2]
bm  = best_model_name
y_actual = np.expm1(y_test)/1e3
y_pred   = np.expm1(all_metrics[bm]['test_pred'])/1e3
sample_i = np.random.choice(len(y_actual), 3000)
ax5.scatter(y_actual[sample_i], y_pred[sample_i],
            alpha=0.25, s=8, color=C3)
lim = max(y_actual.max(), y_pred.max())
ax5.plot([0,lim],[0,lim], color=C4, linewidth=1.5, linestyle='--', label='Perfect')
ax5.set_title(f'Actual vs Predicted — {bm}', color='#dcdcff')
ax5.set_xlabel('Actual Salary ($K)'); ax5.set_ylabel('Predicted Salary ($K)')
ax5.set_xlim(0, min(lim, 400)); ax5.set_ylim(0, min(lim, 400))
ax5.legend(framealpha=0.3); ax5.grid(True, alpha=0.3)

# Summary table (top right slot)
ax6 = axes[1, 0]  # repurpose
ax6.axis('off')
tbl_data = [[n,
             f"${all_metrics[n]['rmse']/1e3:.1f}K",
             f"${all_metrics[n]['mae']/1e3:.1f}K",
             f"{all_metrics[n]['r2']:.4f}",
             f"{all_metrics[n]['mape']:.2f}%"]
            for n in model_names]
tbl = ax6.table(cellText=tbl_data,
                colLabels=['Model','RMSE','MAE','R²','MAPE'],
                cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 2.2)
for (r,c), cell in tbl.get_celld().items():
    cell.set_facecolor('#1a1a30' if r > 0 else '#252550')
    cell.set_edgecolor('#4040607'); cell.set_text_props(color='white')
# Highlight best
best_row = model_names.index(best_model_name) + 1
for c in range(5):
    tbl[best_row, c].set_facecolor('#1a3020')
ax6.set_title('Test Metrics Summary (🥇 = best)', color='#dcdcff', pad=10)

ax7 = axes[1, 1]
ax7.axis('off')
# Run IDs for reference
run_text = "MLflow Run IDs
" + "-"*40
for n, rid in all_run_ids.items():
    run_text += f"
{n}: {rid[:16]}..."
ax7.text(0.05, 0.5, run_text, transform=ax7.transAxes,
         fontsize=9, color='#dcdcff', va='center',
         fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#181830', edgecolor='#404060'))
ax7.set_title('Run IDs', color='#dcdcff')

plt.tight_layout()
plt.savefig('mlflow_results.png', dpi=120, bbox_inches='tight',
            facecolor='#080812', edgecolor='none')
plt.show()
print("✅ MLflow results saved → mlflow_results.png")
print(f"\nView UI: mlflow ui --backend-store-uri {MLFLOW_URI}")

## 🌟 6. Feature Importance & Residual Analysis

In [ ]:
# XGBoost feature importance
xgb_model  = all_metrics['XGBoost']['model']
rf_model   = all_metrics['RandomForest']['model']

ohe_names  = preprocessor.named_transformers_['cat']['ohe']                          .get_feature_names_out(CAT_COLS)
feat_names = np.array(NUM_COLS + list(ohe_names))

fig, axes  = plt.subplots(1, 3, figsize=(20, 7))
fig.suptitle('🌟 Feature Importance & Residual Diagnostics',
             fontsize=14, color=C1, fontweight='bold')

# XGBoost importance
ax1 = axes[0]
xgb_imp    = xgb_model.feature_importances_
top_idx    = np.argsort(xgb_imp)[-20:]
ax1.barh(feat_names[top_idx], xgb_imp[top_idx],
         color=C3, alpha=0.85, edgecolor='white', linewidth=0.4)
ax1.set_title('XGBoost — Top 20 Features', color='#dcdcff')
ax1.set_xlabel('Feature Importance'); ax1.grid(True, alpha=0.3, axis='x')

# RF importance
ax2 = axes[1]
rf_imp  = rf_model.feature_importances_
top_rf  = np.argsort(rf_imp)[-20:]
ax2.barh(feat_names[top_rf], rf_imp[top_rf],
         color=C1, alpha=0.85, edgecolor='white', linewidth=0.4)
ax2.set_title('Random Forest — Top 20 Features', color='#dcdcff')
ax2.set_xlabel('Feature Importance'); ax2.grid(True, alpha=0.3, axis='x')

# Residual plot (XGBoost)
ax3     = axes[2]
resid   = y_test - all_metrics['XGBoost']['test_pred']
ax3.scatter(all_metrics['XGBoost']['test_pred'], resid,
            alpha=0.2, s=6, color=C3)
ax3.axhline(0, color=C4, linewidth=1.5, linestyle='--')
ax3.set_title('XGBoost — Residuals vs Fitted', color='#dcdcff')
ax3.set_xlabel('Fitted (log scale)'); ax3.set_ylabel('Residual')
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('feature_importance.png', dpi=120, bbox_inches='tight',
            facecolor='#080812', edgecolor='none')
plt.show()
print("✅ Feature importance saved → feature_importance.png")

## 🔄 7. Prefect — Automated Retraining Pipeline

Prefect orchestrates ML workflows as **Flows** with **Tasks**.  
When new data arrives, this pipeline automatically retrains and registers a new model.

```
prefect_retrain_flow
  ├── @task load_data()             → loads CSV, returns DataFrame
  ├── @task preprocess()            → apply pipeline, split train/val/test
  ├── @task train_model()           → XGBoost fit + MLflow logging
  ├── @task evaluate_model()        → metrics computation
  ├── @task register_model()        → push to MLflow Model Registry
  └── @task send_alert()            → notify if MAPE > threshold
```

🗣 **Tamil:** Prefect workflow-ஐ automate செய்கிறது. புதிய data வந்தால் automatically retrain, evaluate, register செய்யும். Manual intervention தேவையில்லை.

In [ ]:
import joblib

# Save best model + preprocessor for Prefect to use
best_model_obj = all_metrics[best_model_name]['model']
joblib.dump(best_model_obj,  'best_model.joblib')
joblib.dump(preprocessor,    'preprocessor.joblib')
print(f"Saved best model ({best_model_name}) and preprocessor")

# ── Prefect tasks ─────────────────────────────────────────────────────────────
@task(name="load-data")
def load_data(path: str) -> pd.DataFrame:
    logger = get_run_logger()
    df = pd.read_csv(path)
    logger.info(f"Loaded {len(df):,} rows from {path}")
    return df

@task(name="preprocess-data")
def preprocess_data(df: pd.DataFrame):
    logger = get_run_logger()
    X = df[NUM_COLS + CAT_COLS]
    y = np.log1p(df[TARGET].values)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
    prep = joblib.load('preprocessor.joblib')
    X_tr_pp = prep.fit_transform(X_tr)
    X_te_pp = prep.transform(X_te)
    logger.info(f"Preprocessed: train={X_tr_pp.shape}, test={X_te_pp.shape}")
    return X_tr_pp, X_te_pp, y_tr, y_te

@task(name="train-xgboost")
def train_xgboost(X_train, y_train, X_val, y_val):
    logger = get_run_logger()
    params = dict(n_estimators=400, max_depth=6, learning_rate=0.05,
                  subsample=0.8, colsample_bytree=0.8, random_state=42,
                  early_stopping_rounds=20, eval_metric='rmse',
                  verbosity=0, n_jobs=-1)
    model = xgb.XGBRegressor(**params)
    model.fit(X_train, y_train,
              eval_set=[(X_val, y_val)], verbose=False)
    logger.info(f"Trained XGBoost — best iter: {model.best_iteration}")
    return model

@task(name="evaluate-model")
def evaluate_model(model, X_test, y_test):
    logger = get_run_logger()
    pred   = model.predict(X_test)
    m      = salary_metrics(y_test, pred)
    logger.info(f"RMSE=${m['rmse']/1e3:.1f}K  MAE=${m['mae']/1e3:.1f}K  "
                f"R²={m['r2']:.4f}  MAPE={m['mape']:.2f}%")
    return m

@task(name="register-model")
def register_to_mlflow(model, metrics: dict, run_name: str = 'prefect-retrain'):
    with mlflow.start_run(run_name=run_name):
        mlflow.log_metrics({
            'rmse': metrics['rmse'], 'mae': metrics['mae'],
            'r2':   metrics['r2'],   'mape': metrics['mape'],
        })
        mlflow.log_param('trigger', 'prefect_retrain')
        mlflow.sklearn.log_model(model, 'model',
                                 registered_model_name=MODEL_NAME)
    return "Registered ✅"

@task(name="quality-gate")
def quality_gate(metrics: dict, mape_threshold: float = 25.0):
    logger = get_run_logger()
    if metrics['mape'] > mape_threshold:
        logger.warning(f"⚠️  MAPE={metrics['mape']:.2f}% > threshold {mape_threshold}%")
        return False
    logger.info(f"✅ Quality gate PASSED — MAPE={metrics['mape']:.2f}%")
    return True

# ── Define Prefect flow ───────────────────────────────────────────────────────
@flow(name="salary-predictor-retrain",
      description="Automated salary model retraining pipeline")
def retrain_flow(data_path: str = 'so_survey_synthetic.csv',
                 mape_threshold: float = 25.0):
    logger = get_run_logger()
    logger.info("🚀 Starting retraining flow ...")

    df_raw                           = load_data(data_path)
    X_tr, X_te, y_tr, y_te          = preprocess_data(df_raw)
    n_val                            = max(100, len(X_tr)//10)
    X_val_pf, X_tr_pf = X_tr[:n_val], X_tr[n_val:]
    y_val_pf, y_tr_pf = y_tr[:n_val], y_tr[n_val:]

    model                            = train_xgboost(X_tr_pf, y_tr_pf,
                                                      X_val_pf, y_val_pf)
    metrics                          = evaluate_model(model, X_te, y_te)
    passed                           = quality_gate(metrics, mape_threshold)

    if passed:
        status = register_to_mlflow(model, metrics)
        logger.info(f"Pipeline complete: {status}")
    else:
        logger.error("Pipeline FAILED quality gate — model NOT registered")

    return metrics

# ── Run the flow ──────────────────────────────────────────────────────────────
print("⏳ Running Prefect retraining flow ...")
result = retrain_flow()
print(f"\n✅ Prefect flow complete")
print(f"   RMSE  = ${result['rmse']/1e3:.1f}K")
print(f"   MAE   = ${result['mae']/1e3:.1f}K")
print(f"   R²    = {result['r2']:.4f}")
print(f"   MAPE  = {result['mape']:.2f}%")

## 🚀 8. FastAPI — REST API Service

Production REST API with:
- **Pydantic** request/response models (auto-validation)
- **/predict** endpoint for salary prediction
- **/health** endpoint for readiness checks
- **/metrics** endpoint for monitoring
- Auto-generated Swagger docs at `/docs`

🗣 **Tamil:** FastAPI REST endpoint உருவாக்குகிறது. Pydantic input validation செய்யும். /docs URL-ல் interactive API documentation தானாக generate ஆகும்.

In [ ]:
api_lines = ['# ── app.py — FastAPI Salary Predictor Service ─────────────────────────', 'from fastapi import FastAPI, HTTPException', 'from pydantic import BaseModel, Field, validator', 'from typing   import Optional', 'import numpy  as np', 'import joblib, time, json', 'from pathlib  import Path', '', 'app = FastAPI(', "    title       = 'Salary Predictor API',", "    description = 'Predict annual developer salary from survey features',", "    version     = '1.0.0',", ')', '', '# ── Load model on startup ────────────────────────────────────────────────', "MODEL_PATH = Path('best_model.joblib')", "PREP_PATH  = Path('preprocessor.joblib')", 'model, preprocessor = None, None', 'request_count = 0', 'error_count   = 0', 'start_time    = time.time()', '', "@app.on_event('startup')", 'async def load_model():', '    global model, preprocessor', '    model        = joblib.load(MODEL_PATH)', '    preprocessor = joblib.load(PREP_PATH)', "    print('✅ Model and preprocessor loaded')", '', '# ── Request schema ───────────────────────────────────────────────────────', 'class PredictRequest(BaseModel):', '    YearsCodePro  : Optional[float] = Field(None,  ge=0,   le=50)', '    Country       : str             = Field(...)', '    EdLevel       : str             = Field(...)', '    DevType       : str             = Field(...)', '    OrgSize       : str             = Field(...)', "    Employment    : str             = Field('Full-time')", "    RemoteWork    : str             = Field('Hybrid')", '    Age           : int             = Field(30,     ge=18,  le=80)', '    WorkWeekHrs   : Optional[float] = Field(40.0,  ge=1,   le=100)', '    LanguagesUsed : Optional[float] = Field(3.0,   ge=1,   le=20)', '', 'class PredictResponse(BaseModel):', '    predicted_salary_usd : float', '    confidence_interval  : dict', "    model_version        : str = '1.0.0'", '', '# ── Endpoints ────────────────────────────────────────────────────────────', "@app.get('/health')", 'async def health():', "    return {'status': 'healthy', 'model_loaded': model is not None,", "            'uptime_sec': round(time.time()-start_time, 1)}", '', "@app.get('/metrics')", 'async def metrics():', "    return {'requests_total': request_count, 'errors_total': error_count,", "            'uptime_sec': round(time.time()-start_time, 1)}", '', "@app.post('/predict', response_model=PredictResponse)", 'async def predict(req: PredictRequest):', '    global request_count, error_count', '    request_count += 1', '    try:', '        import pandas as pd', '        row = pd.DataFrame([req.dict()])', '        X   = preprocessor.transform(row)', '        log_pred  = float(model.predict(X)[0])', '        salary    = float(np.expm1(log_pred))', '        low, high = salary * 0.85, salary * 1.15', '        return PredictResponse(', '            predicted_salary_usd = round(salary, 2),', "            confidence_interval  = {'low': round(low,2), 'high': round(high,2)},", '        )', '    except Exception as e:', '        error_count += 1', '        raise HTTPException(status_code=422, detail=str(e))', '', "@app.get('/')", 'async def root():', "    return {'message': 'Salary Predictor API v1.0', 'docs': '/docs'}", '', "if __name__ == '__main__':", '    import uvicorn', "    uvicorn.run(app, host='0.0.0.0', port=8000, reload=False)"]
with open('app.py', 'w') as _f:
    _f.write('\n'.join(api_lines))
print('✅ FastAPI app written → app.py')
print('Run: uvicorn app:app --host 0.0.0.0 --port 8000')
print('Docs: http://localhost:8000/docs')

## 🐳 9. Docker — Containerisation

Package the FastAPI service into a container for reproducible deployment anywhere.

```
salary-predictor/
  ├── app.py                  ← FastAPI service
  ├── best_model.joblib       ← trained model
  ├── preprocessor.joblib     ← fitted preprocessor
  ├── Dockerfile              ← container build instructions
  ├── docker-compose.yml      ← multi-service orchestration
  └── requirements.txt        ← pinned dependencies
```

🗣 **Tamil:** Docker container-ல் app, model, dependencies எல்லாம் pack ஆகும். "My machine-ல் work ஆகுது" problem தீர்க்கும் — எங்கும் same environment.

In [ ]:
df_lines   = ['# ── Dockerfile ────────────────────────────────────────────────────────────', 'FROM python:3.11-slim', '', '# Set work directory', 'WORKDIR /app', '', '# Install system deps', 'RUN apt-get update && apt-get install -y --no-install-recommends \\', '    build-essential curl && rm -rf /var/lib/apt/lists/*', '', '# Copy requirements first (layer caching)', 'COPY requirements.txt .', 'RUN pip install --no-cache-dir -r requirements.txt', '', '# Copy application files', 'COPY app.py .', 'COPY best_model.joblib .', 'COPY preprocessor.joblib .', '', '# Expose port', 'EXPOSE 8000', '', '# Healthcheck', 'HEALTHCHECK --interval=30s --timeout=10s --retries=3 \\', '    CMD curl -f http://localhost:8000/health || exit 1', '', '# Run API', 'CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000", "--workers", "2"]']
dc_lines   = ['# ── docker-compose.yml ─────────────────────────────────────────────────────', "version: '3.8'", 'services:', '  salary-api:', '    build: .', '    ports:', "      - '8000:8000'", '    volumes:', '      - ./mlruns:/app/mlruns', '    environment:', '      - MODEL_PATH=/app/best_model.joblib', '      - MLFLOW_TRACKING_URI=/app/mlruns', '    restart: unless-stopped', '    healthcheck:', "      test: ['CMD', 'curl', '-f', 'http://localhost:8000/health']", '      interval: 30s', '      timeout: 10s', '      retries: 3', '', '  mlflow-ui:', '    image: python:3.11-slim', "    command: bash -c 'pip install mlflow -q && mlflow ui --host 0.0.0.0 --port 5000 --backend-store-uri /mlruns'", '    ports:', "      - '5000:5000'", '    volumes:', '      - ./mlruns:/mlruns', '    restart: unless-stopped']
req_lines  = ['fastapi==0.109.0', 'uvicorn[standard]==0.27.0', 'scikit-learn==1.4.0', 'xgboost==2.0.3', 'joblib==1.3.2', 'pandas==2.2.0', 'numpy==1.26.3', 'pydantic==2.6.0', 'mlflow==2.10.2', 'prefect==2.16.5']
with open('Dockerfile','w') as _f:
    _f.write('\n'.join(df_lines))
with open('docker-compose.yml','w') as _f:
    _f.write('\n'.join(dc_lines))
with open('requirements.txt','w') as _f:
    _f.write('\n'.join(req_lines))
print('✅ Docker files written:')
print('   Dockerfile, docker-compose.yml, requirements.txt')
print()
print('Build & run:')
print('  docker build -t salary-predictor .')
print('  docker-compose up -d')
print('  curl http://localhost:8000/health')

## 🔧 10. GitHub Actions — CI/CD Pipeline

Automates **test → build → deploy** on every push to `main`.

```yaml
On: push → main
  ├── Test job
  │     ├── pytest tests/
  │     └── flake8 lint
  ├── Build job (needs: test)
  │     ├── docker build
  │     └── docker push → DockerHub / ECR
  └── Deploy job (needs: build)
        ├── SSH to EC2
        ├── docker pull latest
        └── docker-compose up -d
```

🗣 **Tamil:** GitHub Actions push ஆனவுடன் automatically test → build → deploy செய்யும். Human intervention தேவையில்லை. CI/CD = Continuous Integration / Continuous Deployment.

In [ ]:
import os
cicd_lines = ['# .github/workflows/ml-cicd.yml', 'name: ML CI/CD Pipeline', '', 'on:', '  push:', '    branches: [main]', '  pull_request:', '    branches: [main]', '', 'env:', '  REGISTRY: ghcr.io', '  IMAGE_NAME: salary-predictor', '', 'jobs:', '  test:', '    runs-on: ubuntu-latest', '    steps:', '      - uses: actions/checkout@v4', '', '      - name: Set up Python', '        uses: actions/setup-python@v5', '        with:', "          python-version: '3.11'", '', '      - name: Install dependencies', '        run: |', '          pip install -r requirements.txt', '          pip install pytest pytest-cov flake8', '', '      - name: Lint', '        run: flake8 app.py --max-line-length=100 --ignore=E501', '', '      - name: Run tests', '        run: pytest tests/ -v --cov=app --cov-report=xml', '', '      - name: Upload coverage', '        uses: codecov/codecov-action@v3', '', '  build-push:', '    needs: test', '    runs-on: ubuntu-latest', "    if: github.ref == 'refs/heads/main'", '    steps:', '      - uses: actions/checkout@v4', '', '      - name: Log in to registry', '        uses: docker/login-action@v3', '        with:', '          registry: ghcr.io', '          username: ${{ github.actor }}', '          password: ${{ secrets.GITHUB_TOKEN }}', '', '      - name: Build and push', '        uses: docker/build-push-action@v5', '        with:', '          push: true', '          tags: ghcr.io/${{ github.repository }}/salary-predictor:latest', '', '  deploy:', '    needs: build-push', '    runs-on: ubuntu-latest', '    steps:', '      - name: Deploy to EC2', '        uses: appleboy/ssh-action@master', '        with:', '          host: ${{ secrets.EC2_HOST }}', '          username: ec2-user', '          key: ${{ secrets.EC2_SSH_KEY }}', '          script: |', '            cd /app/salary-predictor', '            docker-compose pull', '            docker-compose up -d --force-recreate', '            docker system prune -f', "            echo 'Deployment complete'"]
os.makedirs('.github/workflows', exist_ok=True)
with open('.github/workflows/ml-cicd.yml','w') as _f:
    _f.write('\n'.join(cicd_lines))
print('✅ GitHub Actions workflow written → .github/workflows/ml-cicd.yml')
print()
print('Secrets to add in GitHub repo settings:')
print('  EC2_HOST     — public IP of EC2 instance')
print('  EC2_SSH_KEY  — private SSH key content')

## 📡 11. Model Monitoring — Data Drift & Performance Tracking

Monitor two types of issues in production:

| Issue | Detection Method |
|---|---|
| **Data Drift** | Compare feature distributions (KS test, PSI) |
| **Concept Drift** | Track RMSE / MAPE over time windows |
| **Model Degradation** | Alert when metrics exceed thresholds |

🗣 **Tamil:** Production-ல் data distribution மாறலாம் (data drift). Model accuracy குறையலாம் (concept drift). இரண்டையும் monitor செய்து alert அனுப்புவது MLOps-இன் key part.

In [ ]:
from scipy.stats import ks_2samp

def compute_psi(expected, actual, buckets=10):
    expected = np.array(expected)
    actual   = np.array(actual)
    mn, mx   = min(expected.min(), actual.min()), max(expected.max(), actual.max())
    bins     = np.linspace(mn, mx, buckets+1)
    e_cnt    = np.histogram(expected, bins=bins)[0] / len(expected) + 1e-8
    a_cnt    = np.histogram(actual,   bins=bins)[0] / len(actual)   + 1e-8
    psi      = np.sum((e_cnt - a_cnt) * np.log(e_cnt / a_cnt))
    return float(psi)

# Simulate production drift: 30 days, gradually shifting salary distribution
np.random.seed(99)
n_days = 30
monitoring_log = []

# Baseline (train) feature stats
train_exp_mean  = float(X_train['YearsCodePro'].mean())
train_exp_std   = float(X_train['YearsCodePro'].std())
train_salary    = df['ConvertedCompYearly'].values[:len(y_test)]

best_model_obj = all_metrics[best_model_name]['model']

for day in range(n_days):
    # Simulate gradual drift: more senior devs + salary inflation
    drift_factor = 1 + day * 0.01
    n_day        = 300
    day_df       = df.sample(n=n_day, random_state=day).copy()
    day_df['YearsCodePro'] = (day_df['YearsCodePro'].fillna(5) * drift_factor
                               ).clip(0, 40)
    day_df['ConvertedCompYearly'] = (day_df['ConvertedCompYearly']
                                     * (1 + day * 0.008))

    day_X = day_df[NUM_COLS + CAT_COLS]
    day_y = np.log1p(day_df['ConvertedCompYearly'].values)
    day_X_pp = preprocessor.transform(day_X)
    day_pred = best_model_obj.predict(day_X_pp)

    mape_day = np.mean(np.abs((np.expm1(day_y) - np.expm1(day_pred))
                               / np.expm1(day_y))) * 100

    # KS test on YearsCodePro
    ks_stat, ks_p = ks_2samp(
        X_train['YearsCodePro'].dropna().values,
        day_df['YearsCodePro'].dropna().values)

    # PSI on YearsCodePro
    psi = compute_psi(
        X_train['YearsCodePro'].dropna().values,
        day_df['YearsCodePro'].dropna().values)

    monitoring_log.append({
        'day': day+1, 'mape': mape_day,
        'ks_stat': ks_stat, 'ks_pval': ks_p, 'psi': psi,
        'drift_alert': (ks_p < 0.05) or (psi > 0.10),
        'perf_alert':  mape_day > 22.0,
    })

mon_df = pd.DataFrame(monitoring_log)
print(mon_df[['day','mape','ks_stat','psi',
              'drift_alert','perf_alert']].to_string(index=False))
print(f"\nDrift alerts fired on day: "
      f"{mon_df[mon_df['drift_alert']]['day'].tolist()[:5]}")
print(f"Perf  alerts fired on day: "
      f"{mon_df[mon_df['perf_alert']]['day'].tolist()[:5]}")

# ── Monitoring dashboard ──────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
fig.suptitle('📡 Model Monitoring — 30-Day Production Simulation',
             fontsize=14, color=C1, fontweight='bold')

# MAPE over time
ax1 = axes[0,0]
ax1.plot(mon_df['day'], mon_df['mape'], color=C1, linewidth=2, marker='o', ms=4)
ax1.axhline(22.0, color=C4, linewidth=1.5, linestyle='--', label='Alert threshold (22%)')
alert_days = mon_df[mon_df['perf_alert']]['day']
ax1.scatter(alert_days, mon_df[mon_df['perf_alert']]['mape'],
            color=C4, s=80, zorder=5, label='🚨 Alert')
ax1.set_title('MAPE over Time', color='#dcdcff')
ax1.set_xlabel('Day'); ax1.set_ylabel('MAPE (%)')
ax1.legend(framealpha=0.3); ax1.grid(True, alpha=0.3)

# KS statistic
ax2 = axes[0,1]
ax2.plot(mon_df['day'], mon_df['ks_stat'], color=C3, linewidth=2, marker='s', ms=4)
ax2.axhline(0.05, color=C4, linewidth=1.5, linestyle='--', label='p<0.05 threshold')
ax2.fill_between(mon_df['day'], 0, mon_df['ks_stat'],
                 where=mon_df['drift_alert'], alpha=0.25, color=C4, label='Drift zone')
ax2.set_title('KS Statistic — YearsCodePro', color='#dcdcff')
ax2.set_xlabel('Day'); ax2.set_ylabel('KS Statistic')
ax2.legend(framealpha=0.3); ax2.grid(True, alpha=0.3)

# PSI
ax3 = axes[1,0]
ax3.plot(mon_df['day'], mon_df['psi'], color=C2, linewidth=2, marker='^', ms=4)
ax3.axhline(0.10, color=C3,  linewidth=1, linestyle=':', label='Warning (PSI>0.10)')
ax3.axhline(0.25, color=C4,  linewidth=1.5, linestyle='--', label='Alert (PSI>0.25)')
ax3.fill_between(mon_df['day'], 0, mon_df['psi'],
                 where=mon_df['psi']>0.10, alpha=0.2, color=C3)
ax3.set_title('PSI (Population Stability Index)', color='#dcdcff')
ax3.set_xlabel('Day'); ax3.set_ylabel('PSI')
ax3.legend(framealpha=0.3); ax3.grid(True, alpha=0.3)

# Alert timeline
ax4 = axes[1,1]
ax4.scatter(mon_df['day'], [1]*len(mon_df), c=mon_df['drift_alert'].map({True:1,False:0}),
            cmap='RdYlGn_r', s=120, vmin=0, vmax=1, zorder=3,
            label='Drift Alert')
ax4.scatter(mon_df['day'], [0]*len(mon_df), c=mon_df['perf_alert'].map({True:1,False:0}),
            cmap='RdYlGn_r', s=120, vmin=0, vmax=1, zorder=3,
            marker='s', label='Perf Alert')
ax4.set_yticks([0,1]); ax4.set_yticklabels(['Perf','Drift'])
ax4.set_title('Alert Timeline (Red=Alert)', color='#dcdcff')
ax4.set_xlabel('Day'); ax4.grid(True, alpha=0.3, axis='x')
ax4.legend(framealpha=0.3)

plt.tight_layout()
plt.savefig('monitoring_dashboard.png', dpi=120, bbox_inches='tight',
            facecolor='#080812', edgecolor='none')
plt.show()
print("✅ Monitoring dashboard saved → monitoring_dashboard.png")

## 🔎 12. API Prediction Simulation

In [ ]:
# Simulate what the API would return for various developer profiles
profiles = [
    {'YearsCodePro':10,'Country':'USA',    'EdLevel':'Master',   'DevType':'ML engineer',
     'OrgSize':'5000+', 'Employment':'Full-time','RemoteWork':'Fully remote',
     'Age':33,'WorkWeekHrs':42,'LanguagesUsed':5},
    {'YearsCodePro':2, 'Country':'India',  'EdLevel':'Bachelor', 'DevType':'Full-stack',
     'OrgSize':'10-99','Employment':'Full-time','RemoteWork':'Hybrid',
     'Age':24,'WorkWeekHrs':45,'LanguagesUsed':3},
    {'YearsCodePro':15,'Country':'Germany','EdLevel':'PhD',      'DevType':'Data scientist',
     'OrgSize':'1000-4999','Employment':'Full-time','RemoteWork':'Hybrid',
     'Age':40,'WorkWeekHrs':40,'LanguagesUsed':6},
    {'YearsCodePro':5, 'Country':'UK',     'EdLevel':'Bachelor', 'DevType':'DevOps',
     'OrgSize':'100-999','Employment':'Independent contractor','RemoteWork':'Fully remote',
     'Age':30,'WorkWeekHrs':38,'LanguagesUsed':4},
    {'YearsCodePro':0, 'Country':'Brazil', 'EdLevel':'Bootcamp', 'DevType':'Frontend',
     'OrgSize':'1-9',   'Employment':'Part-time','RemoteWork':'In-person',
     'Age':22,'WorkWeekHrs':25,'LanguagesUsed':2},
]

fig, axes = plt.subplots(1, 5, figsize=(22, 6))
fig.suptitle('🔎 FastAPI Prediction Simulation — Developer Salary Estimates',
             fontsize=13, color=C3, fontweight='bold')

for ax, profile in zip(axes, profiles):
    row = pd.DataFrame([profile])
    X_r = preprocessor.transform(row)
    log_pred  = float(best_model_obj.predict(X_r)[0])
    salary    = float(np.expm1(log_pred))
    low, high = salary*0.85, salary*1.15

    bars = ax.bar(['Low','Predicted','High'],
                  [low/1e3, salary/1e3, high/1e3],
                  color=[C2, C1, C4], alpha=0.85, edgecolor='white', linewidth=0.5)
    ax.set_title(
        f"{profile['Country']} | {profile['DevType'][:12]}
"
        f"{profile['EdLevel']} | {profile['YearsCodePro']}yr exp",
        color='#dcdcff', fontsize=8.5)
    ax.set_ylabel('Salary ($K)')
    ax.grid(True, alpha=0.3, axis='y')
    for b, v in zip(bars, [low/1e3, salary/1e3, high/1e3]):
        ax.text(b.get_x()+b.get_width()/2, v+0.5,
                f'${v:.0f}K', ha='center', fontsize=9,
                color='white', fontweight='bold')

plt.tight_layout()
plt.savefig('api_predictions.png', dpi=120, bbox_inches='tight',
            facecolor='#080812', edgecolor='none')
plt.show()
print("✅ API predictions saved → api_predictions.png")

print("\n📋 Sample API responses:")
for p in profiles[:3]:
    row = pd.DataFrame([p])
    X_r = preprocessor.transform(row)
    sal = float(np.expm1(best_model_obj.predict(X_r)[0]))
    print(f"  {p['Country']:10s} {p['DevType']:16s} {p['EdLevel']:12s} "
          f"→  ${sal:,.0f}/yr  "
          f"[${sal*0.85:,.0f} – ${sal*1.15:,.0f}]")

## 🏗️ 13. MLOps Architecture Overview

```
┌──────────────────────────────────────────────────────────────────┐
│                    ML DEVELOPMENT (Notebook)                      │
│  Data → EDA → Feature Eng → Train → Evaluate → Best Model        │
└────────────────────┬─────────────────────────────────────────────┘
                     │ mlflow log_model()
┌────────────────────▼─────────────────────────────────────────────┐
│                    MLFLOW MODEL REGISTRY                          │
│  Staging → Champion → Archived (version control for models)      │
└────────────────────┬─────────────────────────────────────────────┘
                     │ git push → main
┌────────────────────▼─────────────────────────────────────────────┐
│                    GITHUB ACTIONS CI/CD                           │
│  pytest → flake8 → docker build → docker push → SSH deploy       │
└────────────────────┬─────────────────────────────────────────────┘
                     │ docker-compose up
┌────────────────────▼─────────────────────────────────────────────┐
│                    AWS EC2 PRODUCTION                             │
│  FastAPI (port 8000) + MLflow UI (port 5000) + Nginx             │
└────────────────────┬─────────────────────────────────────────────┘
                     │ predictions logged
┌────────────────────▼─────────────────────────────────────────────┐
│                    PREFECT MONITORING FLOW                        │
│  Data Drift (KS/PSI) → Perf Drift (MAPE) → Alert → Retrain      │
└──────────────────────────────────────────────────────────────────┘
```

🗣 **Tamil:** MLOps full pipeline: Data → Train → Track (MLflow) → Package (Docker) → Deploy (EC2) → Monitor (Prefect) → Retrain (automatic).

## 🏆 14. Final MLOps Pipeline Dashboard

In [ ]:
fig = plt.figure(figsize=(20, 12))
fig.suptitle('🏆 Salary Predictor — MLOps Pipeline Dashboard',
             fontsize=17, color=C3, fontweight='bold', y=1.01)
gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.38)

model_names = list(all_metrics.keys())
colors_m    = [C1, C2, C3]

# Model RMSE comparison
ax1 = fig.add_subplot(gs[0,0])
rmses = [all_metrics[n]['rmse']/1e3 for n in model_names]
bars  = ax1.bar(model_names, rmses, color=colors_m, edgecolor='white', lw=0.5, alpha=0.85)
ax1.set_title('Test RMSE ($K) — Lower Better', color='#dcdcff')
ax1.set_ylabel('RMSE ($K)'); ax1.grid(True, alpha=0.3, axis='y')
for b, v in zip(bars, rmses):
    ax1.text(b.get_x()+b.get_width()/2, v+0.3, f'${v:.1f}K',
             ha='center', fontsize=10, color='white', fontweight='bold')
bars[np.argmin(rmses)].set_edgecolor(C4); bars[np.argmin(rmses)].set_linewidth(3)

# R² scores
ax2 = fig.add_subplot(gs[0,1])
r2s  = [all_metrics[n]['r2'] for n in model_names]
bars2 = ax2.bar(model_names, r2s, color=colors_m, edgecolor='white', lw=0.5, alpha=0.85)
ax2.set_title('R² Score — Higher Better', color='#dcdcff')
ax2.set_ylim(0, 1.05); ax2.grid(True, alpha=0.3, axis='y')
for b, v in zip(bars2, r2s):
    ax2.text(b.get_x()+b.get_width()/2, v+0.005, f'{v:.4f}',
             ha='center', fontsize=10, color='white', fontweight='bold')

# Actual vs Pred scatter
ax3 = fig.add_subplot(gs[0,2])
y_act = np.expm1(y_test)/1e3
y_prd = np.expm1(all_metrics[best_model_name]['test_pred'])/1e3
samp  = np.random.choice(len(y_act), 2000)
ax3.scatter(y_act[samp], y_prd[samp], alpha=0.25, s=8, color=C3)
lim   = 300
ax3.plot([0,lim],[0,lim], color=C4, lw=1.5, linestyle='--', label='Perfect')
ax3.set_title(f'Actual vs Predicted ({best_model_name})', color='#dcdcff')
ax3.set_xlabel('Actual ($K)'); ax3.set_ylabel('Predicted ($K)')
ax3.set_xlim(0,lim); ax3.set_ylim(0,lim)
ax3.legend(framealpha=0.3); ax3.grid(True, alpha=0.3)

# Monitoring MAPE trend
ax4 = fig.add_subplot(gs[1,0:2])
ax4.plot(mon_df['day'], mon_df['mape'], color=C1, linewidth=2.5, marker='o', ms=5)
ax4.axhline(22.0, color=C4, linewidth=1.5, linestyle='--', alpha=0.8, label='Alert (22%)')
ax4.fill_between(mon_df['day'], mon_df['mape'], 22.0,
                 where=mon_df['mape']>22.0, alpha=0.25, color=C4)
ax4.set_title('Production MAPE Monitoring — 30 Days', color='#dcdcff')
ax4.set_xlabel('Day'); ax4.set_ylabel('MAPE (%)')
ax4.legend(framealpha=0.3); ax4.grid(True, alpha=0.3)

# PSI trend
ax5 = fig.add_subplot(gs[1,2])
ax5.plot(mon_df['day'], mon_df['psi'], color=C2, linewidth=2, marker='^', ms=5)
ax5.axhline(0.10, color=C3, lw=1, linestyle=':', label='Warning (0.10)')
ax5.axhline(0.25, color=C4, lw=1.5, linestyle='--', label='Alert (0.25)')
ax5.set_title('Data Drift — PSI Score', color='#dcdcff')
ax5.set_xlabel('Day'); ax5.set_ylabel('PSI')
ax5.legend(framealpha=0.3, fontsize=8); ax5.grid(True, alpha=0.3)

# Country salary heatmap
ax6 = fig.add_subplot(gs[2,0:2])
med_c_dev = (df.groupby(['Country','DevType'])['ConvertedCompYearly']
             .median().unstack().fillna(0)/1000)
sns.heatmap(med_c_dev, ax=ax6, cmap='YlOrRd', fmt='.0f', annot=True,
            annot_kws={'size':7.5}, cbar_kws={'label':'Median $K'})
ax6.set_title('Median Salary ($K) — Country × Role', color='#dcdcff')
plt.setp(ax6.get_xticklabels(), rotation=30, ha='right', fontsize=8)
plt.setp(ax6.get_yticklabels(), fontsize=8)

# MLOps stack summary
ax7 = fig.add_subplot(gs[2,2])
ax7.axis('off')
stack_text = (
    "MLOps Stack
" + "─"*30 + "
"
    "Data        → so_survey_synthetic.csv
"
    "Models      → Ridge · RF · XGBoost
"
    f"Best Model  → {best_model_name}
"
    f"Test RMSE   → ${all_metrics[best_model_name]['rmse']/1e3:.1f}K
"
    f"Test R²     → {all_metrics[best_model_name]['r2']:.4f}
"
    "Tracking    → MLflow (./mlruns)
"
    "Orchestrate → Prefect (retrain flow)
"
    "API         → FastAPI (port 8000)
"
    "Container   → Docker + compose
"
    "CI/CD       → GitHub Actions
"
    "Monitor     → KS + PSI + MAPE"
)
ax7.text(0.05, 0.95, stack_text, transform=ax7.transAxes,
         fontsize=9.5, color='#dcdcff', va='top',
         fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='#141430', edgecolor='#404060'))

plt.savefig('mlops_dashboard.png', dpi=120, bbox_inches='tight',
            facecolor='#080812', edgecolor='none')
plt.show()
print("✅ MLOps dashboard saved → mlops_dashboard.png")

SyntaxError: unterminated string literal (detected at line 76) (3495778218.py, line 76)

## 📋 15. Cheat Sheet — Stage 12 Key Concepts

| Concept | Definition | Tamil |
|---|---|---|
| **MLOps** | ML + DevOps — production ML lifecycle | ML-ஐ production-ல் reliably இயக்குவது |
| **MLflow** | Experiment tracking + model registry | Experiments record செய்யும் tool |
| **Run** | One MLflow experiment execution | ஒரு training session record |
| **Artifact** | File logged to MLflow (model, plot) | MLflow-ல் save ஆகும் file |
| **Prefect** | Workflow orchestration (DAG of tasks) | Tasks-ஐ automate செய்யும் tool |
| **Flow** | Prefect pipeline (collection of tasks) | Tasks-இன் தொகுப்பு |
| **FastAPI** | Modern Python REST API framework | Python REST API framework |
| **Pydantic** | Data validation via Python type hints | Input validation library |
| **Docker** | Container — app + deps bundled | App + environment bundle |
| **docker-compose** | Multi-container orchestration | பல containers manage செய்யும் |
| **CI/CD** | Continuous Integration/Deployment | Auto test → build → deploy |
| **Data Drift** | Input distribution shift over time | Data distribution மாற்றம் |
| **PSI** | Population Stability Index (drift metric) | Distribution change அளவு |
| **KS Test** | Kolmogorov-Smirnov distribution test | இரு distributions compare செய்யும் |
| **Model Registry** | Versioned model store (Staging/Prod) | Model versions manage செய்யும் |

---

### MLflow Quick Reference
```python
import mlflow

mlflow.set_tracking_uri('./mlruns')
mlflow.set_experiment('my-experiment')

with mlflow.start_run(run_name='XGBoost-v1') as run:
    mlflow.log_param('n_estimators', 400)
    mlflow.log_metric('rmse', 15000)
    mlflow.log_artifact('feature_plot.png')
    mlflow.sklearn.log_model(model, 'model',
                             registered_model_name='SalaryPredictor')
    print(f"Run ID: {run.info.run_id}")

# Launch UI: mlflow ui --backend-store-uri ./mlruns
```

### Prefect Quick Reference
```python
from prefect import flow, task
from prefect.logging import get_run_logger

@task(name="load-data")
def load_data(path):
    logger = get_run_logger()
    df = pd.read_csv(path)
    logger.info(f"Loaded {len(df)} rows")
    return df

@task(name="train")
def train(X, y):
    model = XGBRegressor().fit(X, y)
    return model

@flow(name="retrain-pipeline")
def retrain():
    df    = load_data("data.csv")
    model = train(df.drop("target",axis=1), df["target"])
    return model

retrain()   # run locally
# retrain.serve(name="scheduled-retrain", cron="0 2 * * *")  # scheduled
```

### FastAPI Quick Reference
```python
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class SalaryRequest(BaseModel):
    YearsCodePro : float
    Country      : str

@app.post("/predict")
async def predict(req: SalaryRequest):
    features = preprocess(req.dict())
    salary   = model.predict(features)[0]
    return {"salary": float(salary)}

# uvicorn app:app --host 0.0.0.0 --port 8000
# curl -X POST http://localhost:8000/predict -H 'Content-Type: application/json' \
#      -d '{"YearsCodePro": 5, "Country": "USA"}'
```

### Docker Quick Reference
```bash
# Build image
docker build -t salary-predictor:latest .

# Run container
docker run -p 8000:8000 salary-predictor:latest

# With compose (API + MLflow UI)
docker-compose up -d

# View logs
docker-compose logs -f salary-api

# Stop
docker-compose down
```

### Data Drift Detection
```python
from scipy.stats import ks_2samp

# KS Test
stat, p_val = ks_2samp(train_data, new_data)
drift = p_val < 0.05   # True = drift detected

# PSI (Population Stability Index)
# PSI < 0.10  → no drift
# PSI 0.10-0.25 → warning
# PSI > 0.25  → significant drift
```

### AWS EC2 Deployment Steps
```bash
# 1. Launch EC2 t3.medium (Amazon Linux 2)
# 2. Install Docker
sudo yum update -y && sudo yum install -y docker
sudo service docker start
sudo curl -L "https://github.com/docker/compose/releases/latest/download/docker-compose-$(uname -s)-$(uname -m)" -o /usr/local/bin/docker-compose
sudo chmod +x /usr/local/bin/docker-compose

# 3. Clone repo & deploy
git clone https://github.com/YOUR_USER/salary-predictor.git
cd salary-predictor
docker-compose up -d

# 4. Open ports 8000, 5000 in EC2 security group
# 5. Access API at http://EC2_IP:8000/docs
```

## 💼 16. Portfolio & Resume Tip

### GitHub README Excerpt
> End-to-end MLOps salary prediction pipeline. Trained Ridge, Random Forest, and XGBoost on a 90K synthetic Stack Overflow Developer Survey. Tracked 3 experiments with MLflow, orchestrated retraining with Prefect, deployed as FastAPI REST service in Docker, with GitHub Actions CI/CD to AWS EC2. Built production monitoring for data drift (KS/PSI) and MAPE degradation.

### Resume Bullet
> - Deployed ML salary predictor as Dockerized FastAPI service on AWS EC2; achieved R²=0.87 with XGBoost; automated retraining with Prefect flows; tracked experiments in MLflow; set up GitHub Actions CI/CD and production data drift monitoring (KS + PSI).

### Portfolio Files to Showcase
| File | Description |
|---|---|
| `so_survey_synthetic.csv` | 90K synthetic developer survey |
| `eda_salary.png` | 7-panel EDA dashboard |
| `mlflow_results.png` | MLflow experiment comparison |
| `feature_importance.png` | XGBoost + RF importance + residuals |
| `monitoring_dashboard.png` | 30-day production monitoring |
| `api_predictions.png` | FastAPI inference for 5 profiles |
| `mlops_dashboard.png` | Complete MLOps pipeline dashboard |
| `app.py` | FastAPI service |
| `Dockerfile` | Container build |
| `docker-compose.yml` | Multi-service orchestration |
| `.github/workflows/ml-cicd.yml` | CI/CD pipeline |
| `mlruns/` | MLflow experiment runs |

### AWS EC2 Free Tier Note
> Use **t2.micro** for free tier (1 vCPU, 1 GB RAM) — sufficient for the FastAPI service.  
> For MLflow UI + API together, **t3.small** (2 GB) is recommended.